# DTAT321. deeptrack.wrappers

<a href="https://colab.research.google.com/github/DeepTrackAI/DeepTrack2/blob/develop/tutorials/3-advanced-topics/DTAT321_wrappers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [1]:
# !pip install deeptrack  # Uncomment if running on Colab/Kaggle.

This advanced tutorial introduces the module `wrappers.py`.


## 1. What is `wrappers.py`?

The `wrappers.py` module provides lightweight containers that keep arrays
together with their metadata. In DeepTrack2, this is useful because an image
or signal is often meaningful only together with information describing how it
was generated, measured, or labeled.

For example, an image may be associated with particle positions, microscope
settings, acquisition parameters, labels, or simulation identifiers. A
`Wrapper` stores the array and these properties in one object, while still
behaving similarly to the underlying NumPy or PyTorch array.


The key roles of `wrappers.py` are:

- **Array Container with Metadata:**
  A `Wrapper` stores an array together with a dictionary of properties.

- **Array-Like Behavior:**
  A `Wrapper` exposes common array attributes such as `shape` and `ndim`.

- **Metadata Preservation:**
  Arithmetic, comparison, and logical operations act on the wrapped array and
  return a new `Wrapper` with the properties preserved.

- **Backend-Independent Behavior:**
  The same interface can be used with NumPy arrays and PyTorch tensors.


## 2. What is a Wrapper?

A `Wrapper` is an object containing two elements:

1. an array, stored in the attribute `array`; and

2. a dictionary of metadata, stored in the attribute `properties`.

The array contains the numerical data. The properties describe the context in
which the data should be interpreted.


In [2]:
import numpy as np

from deeptrack.wrappers import Wrapper

In [3]:
image = np.arange(9).reshape(3, 3)

wrapped_image = Wrapper(
    image,
    properties={
        "position": (1, 2),
        "label": "example",
    },
)

wrapped_image

Wrapper(array=array([[0, 1, 2],
       [3, 4, 5],
       [6, 7, 8]]), properties={'position': (1, 2), 'label': 'example'})

The wrapped array is available through the attribute `array`, while the
metadata is available through the attribute `properties`.


In [4]:
wrapped_image.array

array([[0, 1, 2],
       [3, 4, 5],
       [6, 7, 8]])

In [5]:
wrapped_image.properties

{'position': (1, 2), 'label': 'example'}

A wrapper also exposes array-like attributes, such as `shape` and `ndim`.
These are read from the underlying array.


In [6]:
wrapped_image.shape

(3, 3)

In [7]:
wrapped_image.ndim

2

## 3. Accessing Properties

Properties can be accessed directly through the dictionary stored in
`properties`. They can also be accessed with the method `get_property()`.

The method `get_property()` first checks whether the requested name is an
attribute of the wrapper. If it is not, it looks for the name in the
properties dictionary.


In [8]:
wrapped_image.get_property("position")

(1, 2)

In [9]:
wrapped_image.get_property("label")

'example'

The attribute fallback is useful because the same method can be used for both
array-like attributes and user-defined metadata.


In [10]:
wrapped_image.get_property("shape")

(3, 3)

In [11]:
wrapped_image.get_property("missing_property", default="not found")

'not found'

## 4. Wrappers Behave Like Arrays

Many operations on a wrapper are forwarded to the underlying array. The result
is returned as a new `Wrapper`.

This makes wrappers convenient in image-processing pipelines: the numerical
operation changes the data, while the metadata remains attached to the
result.


In [12]:
shifted_image = wrapped_image + 10

shifted_image

Wrapper(array=array([[10, 11, 12],
       [13, 14, 15],
       [16, 17, 18]]), properties={'position': (1, 2), 'label': 'example'})

In [13]:
shifted_image.array

array([[10, 11, 12],
       [13, 14, 15],
       [16, 17, 18]])

In [14]:
shifted_image.properties

{'position': (1, 2), 'label': 'example'}

Arithmetic operations preserve the properties of the left-hand wrapper.
The same applies to comparison operations.


In [15]:
mask = wrapped_image > 4

mask

Wrapper(array=array([[False, False, False],
       [False, False,  True],
       [ True,  True,  True]]), properties={'position': (1, 2), 'label': 'example'})

In [16]:
mask.array

array([[False, False, False],
       [False, False,  True],
       [ True,  True,  True]])

In [17]:
mask.properties

{'position': (1, 2), 'label': 'example'}

Logical operations also return wrappers. This is useful when masks should
keep the same contextual information as the image from which they were
computed.


In [18]:
large_values = wrapped_image > 2
narrow_values = wrapped_image < 7

combined_mask = large_values & narrow_values

combined_mask

Wrapper(array=array([[False, False, False],
       [ True,  True,  True],
       [ True, False, False]]), properties={'position': (1, 2), 'label': 'example'})

## 5. Operations Between Wrappers

Wrappers can also be combined with other wrappers. In this case, the operation
is applied to the two underlying arrays.

The properties of the left-hand wrapper are preserved in the result.


In [19]:
first = Wrapper(
    np.ones((3, 3)),
    properties={"source": "first"},
)

second = Wrapper(
    2 * np.ones((3, 3)),
    properties={"source": "second"},
)

combined = first + second

combined

Wrapper(array=array([[3., 3., 3.],
       [3., 3., 3.],
       [3., 3., 3.]]), properties={'source': 'first'})

In [20]:
combined.array

array([[3., 3., 3.],
       [3., 3., 3.],
       [3., 3., 3.]])

In [21]:
combined.properties

{'source': 'first'}

Changing the order of the operands changes which properties are preserved.
The numerical result may be the same for commutative operations, but the
metadata follows the left-hand operand.


In [22]:
reverse_combined = second + first

reverse_combined.properties

{'source': 'second'}

## 6. Copying a Wrapper

The method `copy()` creates a shallow copy of a wrapper. It can also replace
the wrapped array or the properties dictionary.

This is useful when a processing step should create a new wrapper explicitly,
while keeping part of the original information.


In [23]:
scaled = wrapped_image.copy(array=2 * wrapped_image.array)

scaled

Wrapper(array=array([[ 0,  2,  4],
       [ 6,  8, 10],
       [12, 14, 16]]), properties={'position': (1, 2), 'label': 'example'})

In [24]:
scaled.properties

{'position': (1, 2), 'label': 'example'}

The properties can also be replaced. Since this is a shallow copy, nested
objects inside the dictionary are not recursively copied.


In [25]:
renamed = wrapped_image.copy(
    properties={"position": (1, 2), "label": "renamed"},
)

renamed.properties

{'position': (1, 2), 'label': 'renamed'}

## 7. Getting the Raw Array

The method `as_array()` returns the wrapped array. This is equivalent to
accessing the attribute `array`, but can make the intention clearer when the
raw numerical data are needed.


In [26]:
raw_array = wrapped_image.as_array()

raw_array

array([[0, 1, 2],
       [3, 4, 5],
       [6, 7, 8]])

## 8. Example - Annotated Images

We now build a small collection of annotated images. Each image contains a
bright particle represented as a single pixel. The wrapper stores the particle
position and an identifier.


In [27]:
def make_particle_image(
    position,
    image_shape=(32, 32),
):
    """Create an image with one bright pixel."""

    image = np.zeros(image_shape)
    image[position] = 1
    return image

In [28]:
positions = [(8, 10), (16, 15), (24, 21)]

wrapped_images = [
    Wrapper(
        make_particle_image(position),
        properties={"position": position, "id": index},
    )
    for index, position in enumerate(positions)
]

wrapped_images[0]

Wrapper(array=array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]]), properties={'position': (8, 10), 'id': 0})

The metadata can be used together with the image data during analysis or
visualization.


In [29]:
for wrapped in wrapped_images:
    position = wrapped.get_property("position")
    identifier = wrapped.get_property("id")

    print(f"Image {identifier}: particle at {position}")

Image 0: particle at (8, 10)
Image 1: particle at (16, 15)
Image 2: particle at (24, 21)


## 9. Example - Processing While Preserving Metadata

A common workflow is to apply numerical processing to an image while keeping
its metadata. Since wrapper operations return wrappers, the metadata remains
attached to the processed output.


In [30]:
processed_images = [5 * wrapped + 0.1 for wrapped in wrapped_images]

processed_images[0].array

array([[0.1, 0.1, 0.1, ..., 0.1, 0.1, 0.1],
       [0.1, 0.1, 0.1, ..., 0.1, 0.1, 0.1],
       [0.1, 0.1, 0.1, ..., 0.1, 0.1, 0.1],
       ...,
       [0.1, 0.1, 0.1, ..., 0.1, 0.1, 0.1],
       [0.1, 0.1, 0.1, ..., 0.1, 0.1, 0.1],
       [0.1, 0.1, 0.1, ..., 0.1, 0.1, 0.1]])

In [31]:
processed_images[0].properties

{'position': (8, 10), 'id': 0}

The processed image has changed, but the particle position and identifier are
still available.


In [32]:
for wrapped in processed_images:
    print(
        wrapped.get_property("id"),
        wrapped.get_property("position"),
        wrapped.array.max(),
    )

0 (8, 10) 5.1
1 (16, 15) 5.1
2 (24, 21) 5.1


## 10. Example - Filtering by Metadata

Metadata can be used to decide how each array should be processed. In this
example, we select the images whose particles are in the lower half of the
image.


In [33]:
lower_half = [
    wrapped
    for wrapped in wrapped_images
    if wrapped.get_property("position")[0] >= 16
]

[wrapped.get_property("id") for wrapped in lower_half]

[1, 2]

The same pattern can be used for selecting images by label, simulation
parameter, acquisition condition, or any other property stored in the wrapper.


## 11. Example - Backend-Independent Wrappers

Wrappers can store either NumPy arrays or PyTorch tensors. Operations are
applied to the wrapped object and preserve its backend.

The following example uses the DeepTrack backend configuration. If PyTorch is
not installed, only the NumPy part should be run.


In [34]:
import deeptrack as dt
from deeptrack import xp

In [35]:
dt.config.set_backend("numpy")

numpy_array = xp.arange(9, dtype=xp.float32).reshape(3, 3)
numpy_wrapper = Wrapper(
    numpy_array,
    properties={"backend": "numpy"},
)

numpy_result = numpy_wrapper + 1

numpy_result

Wrapper(array=array([[1., 2., 3.],
       [4., 5., 6.],
       [7., 8., 9.]], dtype=float32), properties={'backend': 'numpy'})

In [36]:
type(numpy_result.array)

numpy.ndarray

If PyTorch is available, switching backend changes the type of the wrapped
array, but not the way the wrapper is used.


In [37]:
try:
    dt.config.set_backend("torch")

    torch_array = xp.arange(9, dtype=xp.float32).reshape(3, 3)
    torch_wrapper = Wrapper(
        torch_array,
        properties={"backend": "torch"},
    )

    torch_result = torch_wrapper + 1
    print(torch_result)
    print(type(torch_result.array))

finally:
    dt.config.set_backend("numpy")

Wrapper(array=tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]]), properties={'backend': 'torch'})
<class 'torch.Tensor'>
